# Connect 4 Policy Analysis

This notebook compares the available agents without using `tournament.py`, so the bracket BYE issue and the winner-accounting issue do not affect the results.

Agents included:

- `random`: baseline random policy
- `v0`: `groups/my-solution-v0` (no heuristic version)
- `v1`: `groups/my-solution-v1` (heuristic version)
- `current`: `groups/my-solution` (heuristic + transposition-table agent)

The Mojo agents now accept `depth_obj` at runtime, so the same imported policy can be benchmarked at different depths by setting `policy.search_depth`.

In [ ]:
from __future__ import annotations

import importlib
import itertools
import os
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "connect4" / "connect_state.py").exists(), ROOT

from connect4.connect_state import ConnectState

## Helpers

These helpers play honest matches: the first policy is always Red (`-1`) and the second policy is Yellow (`+1`). For fair comparisons, `match()` alternates who starts.

In [ ]:
@dataclass(frozen=True)
class PolicySpec:
    name: str
    module: str
    cls: str
    source_dir: str | None = None
    mojo_file: str | None = None


POLICIES = {
    "random": PolicySpec("random", "groups.random-group.policy", "RandomPolicy", None, None),
    "v0": PolicySpec("v0", "groups.my-solution-v0.policy", "OhYes", "groups/my-solution-v0", "solution_v0.mojo"),
    "v1": PolicySpec("v1", "groups.my-solution-v1.policy", "OhYes", "groups/my-solution-v1", "solution_v1.mojo"),
    "current": PolicySpec("current", "groups.my-solution.policy", "OhYes", "groups/my-solution", "solution.mojo"),
}
SEARCH_DEPTH = 8


def load_policy_class(spec: PolicySpec):
    module = importlib.import_module(spec.module)
    return getattr(module, spec.cls)


def make_policy(spec: PolicySpec, depth: int | None = None):
    cls = load_policy_class(spec)
    policy = cls()
    if hasattr(policy, "mount"):
        policy.mount()
    if depth is not None and hasattr(policy, "search_depth"):
        policy.search_depth = depth
    return policy


def play_game(first_spec: PolicySpec, second_spec: PolicySpec, depth: int | None = None, max_moves: int = 42):
    first = make_policy(first_spec, depth)
    second = make_policy(second_spec, depth)
    state = ConnectState()
    moves = []

    while not state.is_final() and len(moves) < max_moves:
        policy = first if state.player == -1 else second
        action = int(policy.act(state.board))
        moves.append(action)
        state = state.transition(action)

    winner_color = state.get_winner()
    if winner_color == -1:
        winner = first_spec.name
    elif winner_color == 1:
        winner = second_spec.name
    else:
        winner = "draw"
    return {"first": first_spec.name, "second": second_spec.name, "winner": winner, "moves": len(moves), "sequence": moves}


def match(a: PolicySpec, b: PolicySpec, games: int = 10, depth: int | None = None):
    rows = []
    for i in range(games):
        if i % 2 == 0:
            row = play_game(a, b, depth=depth)
        else:
            row = play_game(b, a, depth=depth)
        row["game"] = i + 1
        row["pair"] = f"{a.name} vs {b.name}"
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_match(df: pd.DataFrame, a: str, b: str):
    counts = df["winner"].value_counts().to_dict()
    total = len(df)
    return {
        "pair": f"{a} vs {b}",
        "a": a,
        "b": b,
        "games": total,
        "a_wins": counts.get(a, 0),
        "b_wins": counts.get(b, 0),
        "draws": counts.get("draw", 0),
        "a_win_rate": counts.get(a, 0) / total,
        "b_win_rate": counts.get(b, 0) / total,
        "avg_moves": df["moves"].mean(),
    }

## Current Repository Comparison

This section compares the versions at their policy default depths. The detected `comptime DEPTH` values are shown for reference, but normal calls now pass `search_depth` at runtime.

In [ ]:
depth_constants = []
for name, spec in POLICIES.items():
    if spec.source_dir and spec.mojo_file:
        text = (ROOT / spec.source_dir / spec.mojo_file).read_text()
        m = re.search(r"comptime\s+DEPTH\s*:\s*Int\s*=\s*(\d+)", text)
        depth_constants.append({"agent": name, "comptime_depth": int(m.group(1)) if m else None})
    else:
        depth_constants.append({"agent": name, "comptime_depth": None})

pd.DataFrame(depth_constants)

In [ ]:
GAMES_PER_PAIR = 10

pair_results = []
game_logs = []

for a_name, b_name in itertools.combinations(POLICIES.keys(), 2):
    a = POLICIES[a_name]
    b = POLICIES[b_name]
    df = match(a, b, games=GAMES_PER_PAIR, depth=SEARCH_DEPTH)
    game_logs.append(df)
    pair_results.append(summarize_match(df, a_name, b_name))

repo_games = pd.concat(game_logs, ignore_index=True)
repo_summary = pd.DataFrame(pair_results)
repo_summary

In [ ]:
plot_df = repo_summary.melt(
    id_vars=["pair"],
    value_vars=["a_win_rate", "b_win_rate"],
    var_name="side",
    value_name="win_rate",
)
plot_df["agent"] = plot_df.apply(lambda r: repo_summary.loc[repo_summary["pair"] == r["pair"], "a"].iloc[0] if r["side"] == "a_win_rate" else repo_summary.loc[repo_summary["pair"] == r["pair"], "b"].iloc[0], axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
for pair, sub in plot_df.groupby("pair"):
    ax.bar([f"{pair}\n{sub.iloc[i]['agent']}" for i in range(len(sub))], sub["win_rate"])
ax.set_ylim(0, 1)
ax.set_ylabel("Win rate")
ax.set_title(f"Pairwise win rates at depth {SEARCH_DEPTH}, {GAMES_PER_PAIR} alternating-start games per pair")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()

## Win Rate Against Random

This isolates the baseline question: how reliably does each non-random agent beat random play?

In [ ]:
random_rows = []
for name in ["v0", "v1", "current"]:
    df = match(POLICIES[name], POLICIES["random"], games=20, depth=SEARCH_DEPTH)
    random_rows.append(summarize_match(df, name, "random"))

random_summary = pd.DataFrame(random_rows)
random_summary[["a", "games", "a_wins", "draws", "a_win_rate", "avg_moves"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(random_summary["a"], random_summary["a_win_rate"], color=["#6b7280", "#2563eb", "#059669"])
ax.set_ylim(0, 1)
ax.set_ylabel("Win rate vs random")
ax.set_title(f"Baseline win rate against random at depth {SEARCH_DEPTH}")
plt.tight_layout()

## Move Latency At Depth 8

This measures `act()` runtime on sampled positions. It is a practical performance proxy: lower latency means the agent can search more within a fixed time budget.

In [ ]:
def collect_positions(first: PolicySpec, second: PolicySpec, games: int = 4, depth: int | None = None):
    positions = []
    for i in range(games):
        a, b = (first, second) if i % 2 == 0 else (second, first)
        p1 = make_policy(a, depth=depth)
        p2 = make_policy(b, depth=depth)
        state = ConnectState()
        while not state.is_final():
            positions.append(state.board.copy())
            policy = p1 if state.player == -1 else p2
            state = state.transition(int(policy.act(state.board)))
    return positions


def latency_benchmark(spec: PolicySpec, positions: list[np.ndarray], repeats: int = 3, depth: int | None = None):
    policy = make_policy(spec, depth=depth)
    samples = []
    for board in positions:
        for _ in range(repeats):
            t0 = time.perf_counter()
            _ = int(policy.act(board))
            samples.append(time.perf_counter() - t0)
    arr = np.array(samples)
    return {
        "agent": spec.name,
        "samples": len(samples),
        "mean_ms": arr.mean() * 1000,
        "median_ms": np.median(arr) * 1000,
        "p95_ms": np.quantile(arr, 0.95) * 1000,
    }


positions = collect_positions(POLICIES["v1"], POLICIES["current"], games=4, depth=SEARCH_DEPTH)
latency_df = pd.DataFrame([latency_benchmark(POLICIES[name], positions, depth=SEARCH_DEPTH) for name in ["v0", "v1", "current"]])
latency_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(latency_df["agent"], latency_df["median_ms"], color=["#6b7280", "#2563eb", "#059669"])
ax.set_ylabel("Median act() latency (ms)")
ax.set_title(f"Policy decision latency at depth {SEARCH_DEPTH} on shared sampled positions")
plt.tight_layout()

## Depth Sweep: v1 vs Current

This section benchmarks `v1` and `current` by setting `policy.search_depth` directly before each timing run.

Use fewer depths or fewer sampled positions if high-depth runs are slow.

In [ ]:
def runtime_depth_latency(agent_name: str, depth: int, positions: list[np.ndarray], repeats: int = 1):
    spec = POLICIES[agent_name]
    policy = make_policy(spec, depth=depth)
    samples = []
    for board in positions:
        for _ in range(repeats):
            t0 = time.perf_counter()
            _ = int(policy.act(board))
            samples.append(time.perf_counter() - t0)
    arr = np.array(samples)
    return {
        "agent": agent_name,
        "depth": depth,
        "samples": len(samples),
        "mean_ms": arr.mean() * 1000,
        "median_ms": np.median(arr) * 1000,
        "p95_ms": np.quantile(arr, 0.95) * 1000,
    }

In [ ]:
depth_ranges = {
    "v1": [2, 4, 6, 8, 10],
    "current": [2, 4, 6, 8, 10, 12, 14, 16],
}
depth_rows = []

for agent_name, depths in depth_ranges.items():
    for depth in depths:
        print(f"benchmarking {agent_name} depth={depth}...")
        depth_rows.append(runtime_depth_latency(agent_name, depth, positions[:12], repeats=1))

depth_latency_df = pd.DataFrame(depth_rows)
depth_latency_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for agent_name, sub in depth_latency_df.groupby("agent"):
    sub = sub.sort_values("depth")
    ax.plot(sub["depth"], sub["median_ms"], marker="o", label=agent_name)
ax.set_xlabel("Runtime search depth")
ax.set_ylabel("Median act() latency (ms)")
ax.set_title("Depth vs performance: heuristic v1 vs current TT agent")
ax.legend()
plt.tight_layout()

## Report Additions: Version Comparison, Latency vs Depth, Decision Time by Move

These cells generate the figures used in the PDF report.
Figures are saved to `document/figures/` alongside the typst source.

**Corrected POLICIES dict** — class names match the actual policy files (v0 → `OhYes`, v1 → `NegamaxHeuristics`, v2 → `NegamaxTranspositionTable`, current → `NegamaxAdaptativeDeepening`).

In [ ]:
from pathlib import Path

POLICIES_REPORT = {
    "random":  PolicySpec("random",  "groups.random-group.policy",   "RandomPolicy"),
    "v0":      PolicySpec("v0",      "groups.my-solution-v0.policy", "OhYes"),
    "v1":      PolicySpec("v1",      "groups.my-solution-v1.policy", "NegamaxHeuristics"),
    "v2":      PolicySpec("v2",      "groups.my-solution-v2.policy", "NegamaxTranspositionTable"),
    "current": PolicySpec("current", "groups.my-solution.policy",    "NegamaxAdaptativeDeepening"),
}

FIGURE_DIR = Path("document/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STYLE = {
    "v0":      {"color": "#6b7280", "label": "v0\n(no heuristic)"},
    "v1":      {"color": "#2563eb", "label": "v1\n(heuristic)"},
    "v2":      {"color": "#d97706", "label": "v2\n(heuristic+TT)"},
    "current": {"color": "#059669", "label": "current\n(TT+adapt.)"},
}

REPORT_DEPTH = 8
print("Policies loaded:", list(POLICIES_REPORT))

### Latency vs Depth: v1 (no TT) vs v2 (TT) vs current (TT + adaptive)

The transposition table avoids re-evaluating positions seen earlier in the search tree.
This cell measures per-move decision time at depths 2–8 for all three versions,
showing both the exponential cost of depth and the TT speedup.

In [ ]:
LATENCY_DEPTHS = [2, 4, 6, 8]
N_POSITIONS    = 20

latency_report_rows = []

for agent_name in ("v1", "v2", "current"):
    spec = POLICIES_REPORT[agent_name]
    for d in LATENCY_DEPTHS:
        print(f"  latency {agent_name} depth={d}...", flush=True)
        policy = make_policy(spec, d)
        state  = ConnectState()
        samples = []
        for _ in range(N_POSITIONS):
            if state.is_final():
                state = ConnectState()
            t0 = time.perf_counter()
            action = int(policy.act(state.board))
            samples.append((time.perf_counter() - t0) * 1000)
            state = state.transition(action)
        arr = np.array(samples)
        latency_report_rows.append({
            "agent": agent_name, "depth": d,
            "median_ms": float(np.median(arr)),
            "p25_ms":    float(np.quantile(arr, 0.25)),
            "p75_ms":    float(np.quantile(arr, 0.75)),
        })

latency_report_df = pd.DataFrame(latency_report_rows)
latency_report_df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
agent_style = {
    "v1":      {"color": STYLE["v1"]["color"],      "label": "v1 (no TT)",       "ls": "--"},
    "v2":      {"color": STYLE["v2"]["color"],      "label": "v2 (TT, static)",  "ls": "-."},
    "current": {"color": STYLE["current"]["color"], "label": "current (TT+adap)","ls": "-"},
}

for agent_name, sub in latency_report_df.groupby("agent"):
    sub = sub.sort_values("depth")
    s = agent_style[agent_name]
    ax.plot(sub["depth"], sub["median_ms"], marker="s", color=s["color"],
            linewidth=2, markersize=7, label=s["label"], linestyle=s["ls"])
    ax.fill_between(sub["depth"], sub["p25_ms"], sub["p75_ms"],
                    alpha=0.12, color=s["color"])

ax.set_xlabel("Search depth")
ax.set_ylabel("Median decision time (ms)")
ax.set_yscale("log")
ax.set_xticks(LATENCY_DEPTHS)
ax.set_title("Decision time vs search depth\n(shaded band = IQR, log scale)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_latency_depth.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_latency_depth.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_latency_depth")

### Version Comparison: v1 vs v2 vs current vs Random

Compares win rate of each version against the random baseline at depth=8 (N=30, alternating starts).
Error bars show ±1 binomial standard deviation (√(p(1−p)/n)).

In [ ]:
import math

VERSION_GAMES = 30

version_rows = []
for name in ("v1", "v2", "current"):
    spec = POLICIES_REPORT[name]
    rnd  = POLICIES_REPORT["random"]
    print(f"  {name} vs random ({VERSION_GAMES} games, depth={REPORT_DEPTH})...", flush=True)
    df = match(spec, rnd, games=VERSION_GAMES, depth=REPORT_DEPTH)
    wins = int((df["winner"] == name).sum())
    p = wins / VERSION_GAMES
    std_err = math.sqrt(p * (1 - p) / VERSION_GAMES) if 0 < p < 1 else 0.0
    version_rows.append({"version": name, "wins": wins, "n": VERSION_GAMES,
                          "win_rate": p, "std_err": std_err})

version_report_df = pd.DataFrame(version_rows)
version_report_df

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))
versions   = version_report_df["version"].tolist()
x          = np.arange(len(versions))
bar_colors = [STYLE[v]["color"] for v in versions]

ax.bar(x, version_report_df["win_rate"], color=bar_colors, width=0.5, alpha=0.85)
for xi, (_, row) in zip(x, version_report_df.iterrows()):
    ax.errorbar(xi, row["win_rate"], yerr=row["std_err"],
                fmt="none", color="black", capsize=6, linewidth=1.5)
    ax.text(xi, min(row["win_rate"] + row["std_err"] + 0.04, 1.1),
            f"{row['wins']}/{row['n']}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([STYLE[v]["label"] for v in versions])
ax.set_ylim(0, 1.18)
ax.set_ylabel("Win rate vs random")
ax.set_title(f"Version comparison (depth={REPORT_DEPTH}, N={VERSION_GAMES} games)\n±1 binomial std error")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_version_comparison.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_version_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_version_comparison")

### Color Analysis: Win Rate as First vs Second Player (v1 vs v2 vs current)

Checks whether performance differs by player color against the random baseline.
Error bars show ±1 binomial std error.

In [ ]:
N_COLOR = 15

color_rows = []
for name in ("v1", "v2", "current"):
    spec = POLICIES_REPORT[name]
    rnd  = POLICIES_REPORT["random"]
    print(f"  color {name}...", flush=True)

    p1_games = [play_game(spec, rnd, depth=REPORT_DEPTH) for _ in range(N_COLOR)]
    p1_wins  = sum(g["winner"] == name for g in p1_games)
    p1_wr    = p1_wins / N_COLOR
    p1_err   = math.sqrt(p1_wr * (1 - p1_wr) / N_COLOR) if 0 < p1_wr < 1 else 0.0

    p2_games = [play_game(rnd, spec, depth=REPORT_DEPTH) for _ in range(N_COLOR)]
    p2_wins  = sum(g["winner"] == name for g in p2_games)
    p2_wr    = p2_wins / N_COLOR
    p2_err   = math.sqrt(p2_wr * (1 - p2_wr) / N_COLOR) if 0 < p2_wr < 1 else 0.0

    color_rows += [
        {"version": name, "role": "First (P1)",  "win_rate": p1_wr, "std_err": p1_err, "wins": p1_wins},
        {"version": name, "role": "Second (P2)", "win_rate": p2_wr, "std_err": p2_err, "wins": p2_wins},
    ]

color_df = pd.DataFrame(color_rows)
color_df

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
versions   = ["v1", "v2", "current"]
roles      = ["First (P1)", "Second (P2)"]
role_color = {"First (P1)": "#1d4ed8", "Second (P2)": "#b45309"}
x = np.arange(len(versions))
w = 0.35

for j, role in enumerate(roles):
    sub = color_df[color_df["role"] == role].set_index("version").reindex(versions)
    off = (j - 0.5) * w
    ax.bar(x + off, sub["win_rate"], w, label=role,
           color=role_color[role], alpha=0.82)
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.errorbar(x[i] + off, row["win_rate"], yerr=row["std_err"],
                    fmt="none", color="black", capsize=4, linewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels([STYLE[v]["label"] for v in versions])
ax.set_ylim(0, 1.15)
ax.set_ylabel(f"Win rate vs random (N={N_COLOR} per color)")
ax.set_title("Win rate by player color: v1 vs v2 vs current\n(±1 binomial std error)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_color_analysis.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_color_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_color_analysis")

### Decision Time by Move Number: v1 vs v2 vs current

Tracks how per-move decision time evolves as a game progresses (measured by pieces already on the board).
In the opening (few pieces) the TT is cold; by the mid-game it accumulates hits, reducing redundant search.
This shows where the adaptive deepening mechanism has the most room to exploit time savings.

In [ ]:
N_TIMING_GAMES = 10  # games per agent

def time_by_pieces(spec: PolicySpec, opp_spec: PolicySpec, n_games: int = 10, depth: int = 8):
    """Record spec's decision time vs pieces-on-board.  Spec always plays as P1 (Red)."""
    records = []
    for _ in range(n_games):
        policy = make_policy(spec,     depth)
        opp    = make_policy(opp_spec, depth)
        state  = ConnectState()
        p1_turn = True  # spec is P1 (Red = state.player == -1)
        while not state.is_final():
            n_pieces = int(np.sum(state.board != 0))
            if state.player == -1:       # spec's turn
                t0 = time.perf_counter()
                action = int(policy.act(state.board))
                records.append({"pieces": n_pieces, "ms": (time.perf_counter()-t0)*1000,
                                 "agent": spec.name})
            else:
                action = int(opp.act(state.board))
            state = state.transition(action)
    return pd.DataFrame(records)

timing_frames = []
rnd = POLICIES_REPORT["random"]
for agent_name in ("v1", "v2", "current"):
    print(f"  timing {agent_name}...", flush=True)
    spec = POLICIES_REPORT[agent_name]
    df   = time_by_pieces(spec, rnd, n_games=N_TIMING_GAMES, depth=REPORT_DEPTH)
    timing_frames.append(df)

timing_df = pd.concat(timing_frames, ignore_index=True)
# Bin pieces into groups of 4 for smoother curves
timing_df["piece_bin"] = (timing_df["pieces"] // 4) * 4
timing_summary = (timing_df.groupby(["agent", "piece_bin"])["ms"]
                  .agg(median="median", q25=lambda x: x.quantile(0.25),
                       q75=lambda x: x.quantile(0.75))
                  .reset_index())
timing_summary.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
agent_style2 = {
    "v1":      {"color": STYLE["v1"]["color"],      "label": "v1 (no TT)",        "ls": "--"},
    "v2":      {"color": STYLE["v2"]["color"],      "label": "v2 (TT, static d)", "ls": "-."},
    "current": {"color": STYLE["current"]["color"], "label": "current (TT+adap)", "ls": "-"},
}

for agent_name, sub in timing_summary.groupby("agent"):
    sub = sub.sort_values("piece_bin")
    s   = agent_style2[agent_name]
    ax.plot(sub["piece_bin"], sub["median"], marker="o", color=s["color"],
            linewidth=2, markersize=5, label=s["label"], linestyle=s["ls"])
    ax.fill_between(sub["piece_bin"], sub["q25"], sub["q75"],
                    alpha=0.12, color=s["color"])

ax.set_xlabel("Pieces already on board (game progress)")
ax.set_ylabel("Median decision time (ms)")
ax.set_title(f"Decision time by game stage — depth={REPORT_DEPTH}\n(IQR band, spec plays as P1 vs random)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_time_by_move.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_time_by_move.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_time_by_move")

## Same-Depth 8 Notes

For a strict same-depth comparison, instantiate each policy with `depth=8` using `make_policy(spec, depth=8)` or call the helpers with `depth=8`. The remaining `comptime DEPTH` constants are now just defaults/reference values for these benchmark cells.